# 03c -- PD out-of-time / out-of-regime validation

**What this notebook does (plain English):** A model that looks good on the data
it was *trained* on can still fail on *new* loans. The honest test is to train on
one period and check the model on a **different, later** period it has never seen.
We use the three origination years for exactly this:

- **Split A (out-of-time, same conditions):** train on **2007**, test on **2008**
  -- both crisis years.
- **Split B (out-of-regime):** train on the **crisis (2007+2008)**, test on the
  **calm 2015** book -- a deliberately harder test across very different conditions.

**No leakage:** for each split the PD model is **re-fitted on the training years
only**, then used to score the held-out year. The pooled all-vintage model is
*not* used here -- that would let the test data sneak into training.

**Headline result:** the model's **rank-ordering holds up** out-of-time (it still
sorts risky from safe), but **risk levels shift sharply across regimes** -- the
crisis book averages ~10% default versus ~2% in the calm year, the score
distribution moves wholesale (high PSI), and a model trained only on calm years
**under-predicts** a downturn. That is exactly why PD models are recalibrated
through the cycle.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and reuse the existing PD model + metrics helpers.
import pandas as pd
from src import models, metrics
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet')

In [3]:
# One split = refit PD on the TRAIN vintages only, then score the held-out test
# vintage (this is what prevents the test data leaking into training).
def evaluate_split(train_years, test_years, label):
    tr = base[base['vintage_year'].isin(train_years)]
    te = base[base['vintage_year'].isin(test_years)]
    model, cols = models.fit_pd(tr)                 # fit on training years ONLY
    tr_pd = models.predict_pd(model, cols, tr)
    te_pd = models.predict_pd(model, cols, te)
    ytr = tr['ever_default'].astype(int)
    yte = te['ever_default'].astype(int)
    detail = {'label': label, 'arrays': (ytr, tr_pd, yte, te_pd)}
    row = {
        'split': label,
        'train_auc': round(metrics.auc(ytr, tr_pd), 4),
        'test_auc': round(metrics.auc(yte, te_pd), 4),
        'train_avg_predicted_pd': round(float(tr_pd.mean()), 4),
        'test_avg_predicted_pd': round(float(te_pd.mean()), 4),
        'test_observed_default_rate': round(float(yte.mean()), 4),
        'psi_train_vs_test': round(metrics.psi(tr_pd, te_pd), 4),
    }
    return row, detail

In [4]:
# Run the two forward splits, plus a clearly-labelled reverse 'what-if'.
splits = [
    ([2007], [2008], 'A) out-of-time, same regime: train 2007 -> test 2008'),
    ([2007, 2008], [2015], 'B) out-of-regime: train crisis 2007+08 -> test calm 2015'),
    ([2015], [2007, 2008], 'C) reverse what-if (NOT a forward test): train calm 2015 -> test crisis'),
]
rows, details = [], []
for tr_y, te_y, label in splits:
    r, d = evaluate_split(tr_y, te_y, label)
    rows.append(r); details.append(d)

In [5]:
# The one comparison table (the saved deliverable).
comparison = pd.DataFrame(rows)[[
    'split', 'train_auc', 'test_auc', 'train_avg_predicted_pd',
    'test_avg_predicted_pd', 'test_observed_default_rate', 'psi_train_vs_test']]
save_csv(comparison, 'output/03c_oot_validation.csv')
comparison

,split,train_auc,test_auc,train_avg_predicted_pd,test_avg_predicted_pd,test_observed_default_rate,psi_train_vs_test
0,"A) out-of-time, same regime: train 2007 -> tes...",0.7730,0.8016,0.1374,0.0869,0.0735,0.2411
1,B) out-of-regime: train crisis 2007+08 -> test...,0.7967,0.6979,0.1055,0.0251,0.0242,1.6339
2,C) reverse what-if (NOT a forward test): train...,0.7227,0.7749,0.0242,0.0736,0.1055,1.3452


In [6]:
# Full discrimination detail (AUC / Gini / KS, train vs test) for the record.
for d in details:
    ytr, tr_pd, yte, te_pd = d['arrays']
    print(d['label'])
    print(f"   train: AUC={metrics.auc(ytr,tr_pd):.3f} Gini={metrics.gini(ytr,tr_pd):.3f} KS={metrics.ks(ytr,tr_pd):.3f}")
    print(f"   test : AUC={metrics.auc(yte,te_pd):.3f} Gini={metrics.gini(yte,te_pd):.3f} KS={metrics.ks(yte,te_pd):.3f}")

A) out-of-time, same regime: train 2007 -> test 2008
   train: AUC=0.773 Gini=0.546 KS=0.412
   test : AUC=0.802 Gini=0.603 KS=0.460
B) out-of-regime: train crisis 2007+08 -> test calm 2015
   train: AUC=0.797 Gini=0.593 KS=0.452
   test : AUC=0.698 Gini=0.396 KS=0.292
C) reverse what-if (NOT a forward test): train calm 2015 -> test crisis
   train: AUC=0.723 Gini=0.445 KS=0.341
   test : AUC=0.775 Gini=0.550 KS=0.414


## Interpretation (plain English)

- **Discrimination held out-of-time.** Same-regime (Split A) the test AUC even
  edges up (~0.80); out-of-regime (Split B) it dips to ~0.70 but the model still
  clearly **rank-orders** risky loans above safe ones. Sorting power travels across
  periods, weakening across very different conditions.
- **The risk *level* shifts hard across regimes.** The crisis training book averages
  ~10% default versus ~2% in calm 2015, and the score distribution moves wholesale
  (Split B **PSI ~1.6**, far above the 0.25 "material shift" line). The origination
  features (credit score, LTV) absorb much of this, so the crisis model's *average*
  predicted PD on 2015 happens to land near the observed rate -- but that relies on
  the features doing the work; a level calibrated to the crisis would badly
  over-state calm-period losses.
- **The reverse what-if (Split C)** exposes the dangerous direction: a model trained
  only on the calm 2015 book **under-predicts** the crisis (predicted ~7% vs observed
  ~11%) -- the blind spot of a model built only in good times.
- **Takeaway:** rank-ordering travels, but the *level* and *stability* do not. This
  is precisely why PD models are **recalibrated through the cycle** or carry a
  **macro overlay** -- the same lesson the stress test in notebook 07 makes
  quantitatively.